# Módulo IX — IA Generativa na Prática

> Notebook companheiro — código do livro *Ciência de Dados com Python* (Howard Roatti). Licença MIT.


# Módulo IX -- IA Generativa na Prática

## VISÃO GERAL

No Módulo VIII construímos a base conceitual: entendemos a atenção, a arquitetura *Transformer*, a tokenização, os *embeddings* e o que é, afinal, uma LLM. Agora vamos **colocar a mão na massa**. Este módulo é eminentemente prático: ao final dele, você terá conversado com modelos de linguagem a partir de código Python, controlado a forma como eles geram texto, aprendido a arte de escrever bons *prompts* e -- o ponto alto -- construído um sistema capaz de responder perguntas sobre os **seus próprios documentos**, o chamado **RAG**.

Seguindo a decisão pedagógica deste curso, **todos os exemplos executáveis usam modelos de código aberto rodando localmente** (via Hugging Face `transformers` e Ollama), sem necessidade de contratar APIs pagas. Isso lhe dá autonomia total, privacidade dos dados e custo zero de uso -- ideal para aprender e experimentar. Sempre que fizer sentido, mostraremos também, de forma **ilustrativa**, como o mesmo conceito apareceria ao consumir uma API comercial (Anthropic, OpenAI), para que você reconheça o padrão -- mas nada neste módulo depende de uma chave paga.

Ao final deste módulo você será capaz de:

- Preparar um ambiente Python para IA generativa e carregar um modelo aberto localmente;
- Gerar texto controlando os parâmetros de amostragem (*temperature*, *top-p*, *top-k*) e entender matematicamente o que cada um faz;
- Escrever *prompts* eficazes (papéis, *few-shot*, *chain-of-thought*) e obter **saída estruturada** (JSON) confiável;
- Gerar *embeddings*, medir similaridade semântica e indexá-los em um **banco vetorial**;
- Construir um pipeline completo de **RAG** (*Retrieval-Augmented Generation*) do zero;
- Distinguir **RAG** de *fine-tuning* e avaliar as saídas de uma LLM (inclusive com a técnica *LLM-as-judge*).

## Preparando o Ambiente

Retomando as boas práticas do Módulo I (atualizado), trabalharemos em um ambiente virtual isolado, com Python 3.12+. Crie e ative um ambiente e instale as bibliotecas que usaremos ao longo do módulo:

```bash
# criação do ambiente (use venv ou a ferramenta uv, mais rápida)
python -m venv .venv
# Windows: .venv\Scripts\activate    |    Linux/Mac: source .venv/bin/activate

# bibliotecas principais deste módulo
pip install torch transformers sentence-transformers faiss-cpu chromadb
```

Para os exemplos com **Ollama** -- a forma mais simples de rodar um LLM local, sem lidar com detalhes de *hardware* --, instale o aplicativo a partir de <https://ollama.com> e, no terminal, baixe um modelo pequeno (aberto):

```bash
ollama pull llama3.2:3b     # modelo aberto, ~2 GB, roda em CPU/GPU modestas
```

> **Nota sobre *hardware*.** Modelos abertos variam de centenas de milhões a centenas de bilhões de parâmetros. Neste módulo priorizamos modelos **pequenos** (1–3 bilhões de parâmetros), que rodam em um computador comum -- alguns até só com CPU. A qualidade é inferior à dos grandes modelos comerciais, mas é mais do que suficiente para **aprender os conceitos**, que são idênticos independentemente do tamanho do modelo.

## Conversando com uma LLM Localmente

Há dois caminhos práticos para rodar uma LLM na sua máquina. Começaremos pelo mais didático -- a biblioteca `transformers`, que expõe as engrenagens que estudamos no Módulo VIII -- e depois pelo mais conveniente, o Ollama.

### Caminho 1 -- Hugging Face `transformers`

A biblioteca `transformers`, da Hugging Face, é o padrão de fato para trabalhar com modelos *Transformer* abertos. A forma mais direta de usá-la é pela função `pipeline`, que encapsula tokenização, inferência e destokenização -- exatamente as etapas do Módulo VIII:


In [ ]:
from transformers import pipeline

# 'text-generation' carrega um modelo decoder-only (Módulo VIII)
gerador = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",   # modelo aberto, versão 'Instruct' (Módulo VIII)
    device_map="auto",                     # usa GPU se houver; senão, CPU
)

mensagens = [
    {"role": "system", "content": "Você é um tutor de ciência de dados, didático e conciso."},
    {"role": "user", "content": "Explique, em uma frase, o que é overfitting."},
]

saida = gerador(mensagens, max_new_tokens=120)
print(saida[0]["generated_text"][-1]["content"])

Repare em três pontos que conectam com o módulo anterior: (1) escolhemos deliberadamente a versão **`Instruct`** do modelo -- a versão *base* não seguiria bem a instrução; (2) a entrada é uma lista de **mensagens com papéis** (`system`, `user`), o formato de conversa que as LLMs *chat* esperam; (3) `max_new_tokens` limita a geração em **tokens**, não em palavras -- de novo o conceito de token do Módulo VIII.

Para quem quer ver as engrenagens girando, o mesmo resultado pode ser obtido de forma explícita, separando *tokenizer* e modelo:


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

nome = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(nome)
modelo = AutoModelForCausalLM.from_pretrained(nome, device_map="auto", torch_dtype="auto")

# 1) o template de chat formata os papéis no padrão que ESTE modelo espera
entrada = tokenizer.apply_chat_template(mensagens, add_generation_prompt=True, return_tensors="pt")

# 2) o modelo gera tokens de forma autorregressiva (Módulo VIII)
with torch.no_grad():
    gerados = modelo.generate(entrada.to(modelo.device), max_new_tokens=120)

# 3) destokenização: de volta a texto legível
print(tokenizer.decode(gerados[0][entrada.shape[-1]:], skip_special_tokens=True))

Este trecho torna visível tudo o que discutimos no Módulo VIII: o texto vira **tokens** (`apply_chat_template`/`return_tensors`), o modelo os processa e gera novos tokens **um a um** (`generate`, autorregressivo), e o *tokenizer* faz o caminho de volta (`decode`).

### Caminho 2 -- Ollama

O Ollama abstrai a complexidade de carregar pesos e gerenciar *hardware*, expondo um servidor local simples. Depois de `ollama pull`, você pode conversar via Python:


In [ ]:
import ollama

resposta = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {"role": "system", "content": "Você é um tutor de ciência de dados."},
        {"role": "user", "content": "O que é validação cruzada? Responda em 2 frases."},
    ],
)
print(resposta["message"]["content"])

> **Paralelo com APIs comerciais (ilustrativo -- requer chave, não usado neste curso).** Note como a estrutura de "lista de mensagens com papéis" é **universal**. Ao consumir uma API paga, o código teria quase a mesma forma -- por exemplo, com a biblioteca da Anthropic: `client.messages.create(model="claude-...", system="...", messages=[{"role": "user", "content": "..."}])`. O que muda é *onde* o modelo roda e o fato de haver cobrança por token; o **conceito é idêntico**. Por isso, o que você aprende localmente transfere-se diretamente para qualquer provedor.

## Controlando a Geração: Amostragem e Seus Parâmetros

No Módulo VIII dissemos que um modelo *decoder-only* prevê "o próximo token". Sejamos precisos: a cada passo, o modelo produz um vetor de *scores* (chamados **logits**) sobre **todo o vocabulário**, e a *softmax* os converte em uma **distribuição de probabilidade**. A pergunta prática é: dada essa distribuição, **qual token escolher?** As respostas a essa pergunta são os parâmetros de geração -- e entendê-los é o que separa resultados medíocres de resultados excelentes.

### *Temperature*: o Controle de Aleatoriedade

A *softmax* com **temperatura** $T$ é definida como:


onde $z_i$ é o logit do token $i$. O parâmetro $T$ "achata" ou "aguça" a distribuição:

- $T \to 0$: a distribuição se concentra quase toda no token mais provável -- geração **determinística** e conservadora (equivalente a escolher sempre o máximo, o chamado *greedy decoding*). Ideal para tarefas factuais, extração e código.
- $T = 1$: usa a distribuição como o modelo a produziu.
- $T > 1$ (ex.: 1.2–1.5): achata a distribuição, dando mais chance a tokens improváveis -- geração mais **criativa e diversa**, porém mais propensa a erros e incoerências. Útil para *brainstorming* e escrita criativa.

### *Top-k* e *Top-p* (*Nucleus Sampling*)

Amostrar de todo o vocabulário, mesmo com temperatura baixa, permite que tokens absurdos apareçam ocasionalmente. Dois filtros restringem o conjunto de candidatos antes de sortear:

- **Top-k:** mantém apenas os $k$ tokens mais prováveis (ex.: $k=50$) e zera o resto.
- **Top-p (*nucleus sampling*):** mantém o **menor conjunto** de tokens cuja probabilidade acumulada atinge $p$ (ex.: $p=0.9$). É adaptativo: quando o modelo está "certo", poucos tokens bastam; quando está "em dúvida", o conjunto se amplia.

Na prática, combina-se `temperature` com `top_p`. Veja o efeito no código local:


In [ ]:
saida = gerador(
    mensagens,
    max_new_tokens=200,
    do_sample=True,        # habilita amostragem (senão, é greedy)
    temperature=0.7,       # criatividade moderada
    top_p=0.9,             # nucleus sampling
    repetition_penalty=1.1 # penaliza repetição de tokens (evita loops)
)

> **Regra de bolso.** Precisão factual, extração de dados, geração de JSON ou código → **temperatura baixa** (0–0.3). Texto criativo, variações, ideias → **temperatura mais alta** (0.7–1.2). Este é um dos ajustes de maior impacto e menor custo em qualquer aplicação de IA generativa.

## Engenharia de *Prompt* (*Prompt Engineering*)

Como vimos no Módulo VIII, LLMs são extremamente **sensíveis à forma da entrada**. *Prompt engineering* é a disciplina de estruturar essa entrada para obter respostas melhores. Não é "magia": são padrões com fundamento no modo como o modelo foi treinado.

### Papéis: *system* e *user*

A mensagem **`system`** define o comportamento, a persona e as regras gerais ("Você é um assistente jurídico; responda apenas com base nos documentos fornecidos; se não souber, diga que não sabe"). As mensagens **`user`** trazem as solicitações concretas. Separar as duas coisas dá muito mais controle do que amontoar tudo em um único texto.

### *Zero-shot*, *Few-shot* e o Poder dos Exemplos

- **Zero-shot:** pedir a tarefa diretamente, sem exemplos. Funciona bem para tarefas comuns.
- **Few-shot:** incluir **alguns exemplos** de entrada→saída no próprio *prompt*. Isso ativa o *in-context learning* (uma das capacidades emergentes do Módulo VIII): o modelo "pega o padrão" e o aplica ao novo caso, sem nenhum re-treino.


In [ ]:
prompt_fewshot = """Classifique o sentimento como POSITIVO, NEGATIVO ou NEUTRO.

Texto: "Adorei o produto, chegou antes do prazo!"  ->  POSITIVO
Texto: "Veio quebrado e o suporte não respondeu."  ->  NEGATIVO
Texto: "O pacote chegou na terça-feira."           ->  NEUTRO
Texto: "A entrega atrasou, mas o produto é ótimo." ->  """

### *Chain-of-Thought*: Pedir para "Pensar"

Para problemas que exigem raciocínio (matemática, lógica, várias etapas), instruir o modelo a **"pensar passo a passo"** antes de responder melhora drasticamente a acurácia. A razão é intuitiva à luz do Módulo VIII: como o modelo gera token a token, dar-lhe "espaço para raciocinar" permite que ele construa a resposta gradualmente, em vez de tentar acertar de imediato. Muitos modelos modernos de "raciocínio" já fazem isso internamente.

### Saída Estruturada (JSON)

Em aplicações reais, muitas vezes queremos que a saída seja **consumível por um programa**, não por um humano -- por exemplo, um JSON. Duas estratégias:

1. **Instrução clara + exemplo do formato** no *prompt* (com temperatura baixa).
2. **Saída estruturada forçada** (*constrained/structured output*): recursos que **garantem** que a saída obedeça a um esquema. Localmente, bibliotecas como `outlines` fazem isso restringindo os tokens gerados; em Ollama, o parâmetro `format`:


In [ ]:
import ollama, json

resposta = ollama.chat(
    model="llama3.2:3b",
    messages=[{"role": "user",
               "content": "Extraia nome e idade: 'Ana tem 30 anos.' Responda em JSON."}],
    format={  # esquema JSON: garante saída válida e parseável
        "type": "object",
        "properties": {"nome": {"type": "string"}, "idade": {"type": "integer"}},
        "required": ["nome", "idade"],
    },
)
dados = json.loads(resposta["message"]["content"])
print(dados["nome"], dados["idade"])   # -> Ana 30

Saída estruturada é a ponte entre a IA generativa e o software tradicional -- e será essencial no Módulo X, quando os agentes precisarem decidir, de forma parseável, *qual ferramenta chamar*.

## *Embeddings* e Busca Semântica

Retomamos aqui os *embeddings* do Módulo VIII, agora como ferramenta prática. A ideia central: converter textos inteiros (não apenas tokens) em vetores, de modo que **textos com significado próximo tenham vetores próximos**. Com isso, podemos "buscar por significado", e não por palavras exatas -- uma busca por "carro" encontra um documento sobre "automóvel".

A biblioteca `sentence-transformers` torna isso trivial, com modelos abertos:


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

modelo_emb = SentenceTransformer("all-MiniLM-L6-v2")  # modelo aberto, leve

frases = ["O gato dorme no sofá.",
          "Um felino descansa no móvel.",   # semanticamente próxima da 1ª
          "A bolsa de valores caiu hoje."]  # tema totalmente distinto
vetores = modelo_emb.encode(frases)         # matriz (3, 384): 3 frases, 384 dimensões

def cosseno(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(round(cosseno(vetores[0], vetores[1]), 3))  # alta similaridade (~0.7+)
print(round(cosseno(vetores[0], vetores[2]), 3))  # baixa similaridade (~0.1)

A **similaridade do cosseno** -- que já apareceu no Módulo VI ao falarmos de medidas de similaridade -- mede o ângulo entre dois vetores: 1 significa mesma direção (sentido idêntico), 0 significa ortogonais (sem relação). É a métrica padrão para comparar *embeddings*.

### Bancos de Dados Vetoriais

Comparar uma consulta contra 3 vetores é fácil; contra **milhões**, precisamos de estrutura. Um **banco de dados vetorial** armazena *embeddings* e realiza busca por vizinhos mais próximos (*nearest neighbors*) de forma eficiente. Duas opções abertas populares são **FAISS** (biblioteca da Meta, muito rápida) e **Chroma** (banco vetorial com API amigável). Exemplo com Chroma:


In [ ]:
import chromadb

cliente = chromadb.Client()
colecao = cliente.create_collection("meus_docs")

# indexa documentos (o Chroma gera os embeddings internamente)
colecao.add(
    documents=["Python é uma linguagem de programação.",
               "O Pandas serve para análise de dados.",
               "A fotossíntese ocorre nas plantas."],
    ids=["d1", "d2", "d3"],
)

# busca semântica: encontra os 2 documentos mais relevantes
resultado = colecao.query(query_texts=["Como analiso dados tabulares?"], n_results=2)
print(resultado["documents"])   # -> traz o doc do Pandas em primeiro

Com *embeddings* + banco vetorial, temos o mecanismo de **recuperação** (*retrieval*). Falta acoplá-lo à **geração**. É disso que trata o RAG.

## RAG -- *Retrieval-Augmented Generation*

Chegamos ao conceito mais importante deste módulo para aplicações reais. Recorde duas limitações das LLMs vistas no Módulo VIII: elas têm **corte de conhecimento** (não sabem de fatos recentes nem de dados privados) e tendem a **alucinar** (inventar com confiança). O **RAG** ataca as duas de uma vez.

A ideia: em vez de confiar apenas no que está "nos pesos", **recuperamos** trechos relevantes de uma base de conhecimento nossa e os **injetamos no *prompt*** como contexto, instruindo o modelo a responder **com base neles**. O modelo deixa de ser um "sabe-tudo de memória" e passa a ser um "raciocinador sobre documentos fornecidos".

O pipeline de RAG tem duas fases:

**Fase 1 -- Indexação (uma vez, offline):**
1. **Ingestão:** reunir os documentos (PDFs, páginas, planilhas -- retomando o carregamento de arquivos do Módulo IV).
2. **Fragmentação (*chunking*):** quebrar cada documento em pedaços menores (parágrafos ou blocos de ~algumas centenas de tokens), pois recuperamos trechos, não livros inteiros.
3. **Embeddings:** gerar o vetor de cada fragmento (`sentence-transformers`).
4. **Indexação:** guardar os vetores em um banco vetorial.

**Fase 2 -- Consulta (a cada pergunta, online):**
5. **Recuperar:** gerar o *embedding* da pergunta e buscar os $k$ fragmentos mais similares.
6. **Aumentar (*augment*):** montar um *prompt* que inclui os fragmentos recuperados + a pergunta.
7. **Gerar:** a LLM responde usando esse contexto.


Veja um RAG **mínimo e completo**, 100% local, juntando as peças deste módulo:


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np, ollama

# --- Fase 1: indexação ---
base = [
    "A política de reembolso da empresa permite devolução em até 30 dias.",
    "O horário de atendimento é de segunda a sexta, das 9h às 18h.",
    "Produtos em promoção não podem ser trocados, apenas reembolsados.",
]
emb = SentenceTransformer("all-MiniLM-L6-v2")
indice = emb.encode(base)                      # embeddings dos fragmentos

# --- Fase 2: consulta ---
pergunta = "Posso devolver um item comprado há 20 dias?"
q = emb.encode(pergunta)
sims = indice @ q / (np.linalg.norm(indice, axis=1) * np.linalg.norm(q))  # cosseno vetorizado
top = sims.argsort()[::-1][:2]                 # 2 fragmentos mais relevantes
contexto = "\n".join(base[i] for i in top)

prompt = f"""Responda à pergunta usando SOMENTE o contexto abaixo.
Se a resposta não estiver no contexto, diga que não há informação.

Contexto:
{contexto}

Pergunta: {pergunta}"""

resposta = ollama.chat(model="llama3.2:3b",
                       messages=[{"role": "user", "content": prompt}])
print(resposta["message"]["content"])

Este exemplo, com poucas linhas, entrega algo poderoso: um assistente que responde sobre **a sua** base de conhecimento, cita implicitamente a fonte e recusa-se a inventar quando a informação não está lá. É o alicerce dos "chatbots sobre documentos" que se tornaram onipresentes -- e a base sobre a qual construiremos agentes no Módulo X.

### RAG vs. *Fine-tuning*

Uma dúvida frequente: para "ensinar coisas novas" a um modelo, uso RAG ou faço *fine-tuning* (re-treino, Módulo VIII)? A regra prática:


Na prática, muitas soluções **combinam** as duas: *fine-tuning* para o "jeito de falar" e RAG para o "o que saber". Para a maioria dos casos de conhecimento factual, **comece por RAG** -- é mais barato, mais transparente e mais fácil de manter.

## Além do Texto: Multimodalidade (Panorama)

Modelos modernos são cada vez mais **multimodais**: além de texto, processam imagens, áudio e até vídeo, e alguns geram imagens e áudio. Um modelo de **visão-linguagem** (VLM) pode descrever uma foto, ler um gráfico ou extrair dados de um documento escaneado; modelos de difusão (herdeiros conceituais das GANs do Módulo VII) geram imagens a partir de texto. Existem opções abertas (por exemplo, VLMs disponíveis via `transformers` e via Ollama). O tratamento aprofundado foge ao escopo deste módulo introdutório, mas fica o registro de que **os mesmos conceitos -- tokens, atenção, contexto -- estendem-se a outras modalidades**.

## Avaliando as Saídas de uma LLM

Diferentemente dos modelos do Módulo VI, cuja avaliação usava métricas objetivas (acurácia, F1, RMSE), avaliar texto gerado é mais sutil: muitas respostas "certas" são possíveis. Estratégias:

- **Métricas de referência** (quando há uma resposta esperada): sobreposição textual (BLEU, ROUGE) ou -- melhor -- **similaridade semântica** via *embeddings* (reaproveitando o cosseno que já dominamos).
- **Avaliação humana:** ainda o padrão-ouro para qualidade percebida, porém caro e lento.
- ***LLM-as-judge*:** usar uma (outra) LLM como avaliadora, pedindo que pontue a resposta segundo critérios definidos (fidelidade ao contexto, completude, clareza). Escalável e surpreendentemente correlacionado com o julgamento humano, desde que a rubrica seja clara.
- **Métricas específicas de RAG:** *faithfulness* (a resposta se apoia mesmo no contexto recuperado?) e *relevância da recuperação* (os fragmentos trazidos eram os certos?).

> **Alerta metodológico.** *LLM-as-judge* e a própria IA generativa herdam vieses e podem errar. No **Módulo XI** aprofundaremos avaliação sistemática, *guardrails*, segurança (incluindo *prompt injection*) e as questões éticas -- temas indispensáveis antes de levar qualquer solução de IA generativa à produção.

## Síntese do Módulo

Saímos da teoria para a prática. Carregamos LLMs abertas localmente (via `transformers` e Ollama), controlamos sua geração entendendo a matemática de *temperature*, *top-k* e *top-p*, e exercitamos o *prompt engineering* (papéis, *few-shot*, *chain-of-thought*, saída estruturada em JSON). Transformamos textos em *embeddings*, medimos similaridade semântica e os indexamos em bancos vetoriais. Com essas peças, construímos um **RAG** completo -- o padrão que conecta LLMs aos nossos próprios dados -- e aprendemos quando preferi-lo ao *fine-tuning*. Por fim, discutimos como avaliar saídas generativas.

Temos agora tudo de que precisamos para o próximo salto. Uma LLM que sabe conversar, seguir instruções, produzir saída estruturada e consultar uma base de conhecimento está a um passo de se tornar algo mais autônomo: um **agente** capaz de **decidir e agir**, usando ferramentas para interagir com o mundo. É o tema do **Módulo X -- Agentes Inteligentes**.

## Exercícios

1. **Ambiente e primeira geração.** Instale o ambiente descrito e rode o primeiro exemplo com `transformers` (ou Ollama). Peça ao modelo para explicar um conceito visto no Módulo VI e avalie a qualidade da resposta.
2. **Temperatura.** Gere a **mesma** solicitação criativa (ex.: "escreva um slogan para uma cafeteria") com `temperature` 0.2, 0.8 e 1.4, mantendo o resto fixo. Compare a diversidade e a coerência das saídas e relacione o resultado com a fórmula da *softmax* com temperatura.
3. **Few-shot.** Complete o exemplo de classificação de sentimento *few-shot* do módulo e teste-o em 5 frases novas. Depois remova os exemplos (*zero-shot*) e compare a consistência.
4. **Saída estruturada.** Escreva um *prompt* que extraia, de uma frase livre, um JSON com os campos `produto`, `preco` (número) e `em_promocao` (booleano). Valide que a saída é um JSON parseável com `json.loads`.
5. **Busca semântica.** Crie uma pequena base de 10 frases sobre temas variados, gere seus *embeddings* e implemente uma busca que, dada uma consulta, retorne as 3 frases mais similares por cosseno. Explique um caso em que a busca semântica acerta onde a busca por palavra-chave falharia.
6. **RAG (projeto).** Adapte o RAG mínimo do módulo para responder perguntas sobre um documento **real** seu (por exemplo, um PDF de regras ou um capítulo deste curso). Implemente a fragmentação (*chunking*) em pedaços de ~500 tokens e teste 3 perguntas, incluindo uma cuja resposta **não** esteja no documento (o modelo deve recusar-se a inventar).
7. **RAG vs. fine-tuning.** Para cada cenário, indique e justifique a melhor abordagem: (a) um chatbot que precisa responder sobre a política interna da empresa, atualizada mensalmente; (b) um modelo que deve sempre responder no tom formal e no formato de um parecer jurídico.
8. **Avaliação (LLM-as-judge).** Escreva um *prompt* de avaliação que peça a uma LLM para pontuar, de 0 a 5, a "fidelidade ao contexto" de uma resposta de RAG, justificando a nota. Discuta os riscos de confiar nessa avaliação automática.
